# Delta Lake Tutorial

This notebook demonstrates key Delta Lake features on Databricks:

**Core Features:**
* Loading data and creating Delta tables
* ACID transactions (UPDATE operations)
* Time travel and version history
* Managed vs external tables
* Table optimization (OPTIMIZE and VACUUM)

**Dataset:** AdventureWorks product catalog (293 rows)

**Sections:**
1. Data ingestion from CSV
2. Delta table operations and versioning
3. Table types (external vs managed)
4. Metadata inspection
5. Table maintenance operations

In [0]:
import pandas as pd

# Load CSV from URL (using pandas for HTTPS support)
url = "https://raw.githubusercontent.com/MicrosoftLearning/mslearn-databricks/main/data/products.csv"
pdf = pd.read_csv(url)

# Convert to Spark DataFrame
df = spark.createDataFrame(pdf)

print(f"✓ Loaded {df.count()} products")
display(df.limit(10))

✓ Loaded 295 products


ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3399.99
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99


In [0]:
# Save DataFrame as managed Delta table in Unity Catalog
table_name = "default.products_delta"

df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"✓ Delta table saved: {table_name}")

✓ Delta table saved: default.products_delta


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *

# Load Delta table and apply 10% discount to product 771
deltaTable = DeltaTable.forName(spark, "default.products_delta")

deltaTable.update(
    condition = "ProductID == 771",
    set = { "ListPrice": "ListPrice * 0.9" }
)

print("✓ Product 771 updated")
display(deltaTable.toDF().filter("ProductID = 771"))

✓ Product 771 updated


ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3059.991


In [0]:
# View Delta table version history
deltaTable = DeltaTable.forName(spark, "default.products_delta")

history_df = deltaTable.history(10).select("version", "timestamp", "operation", "operationParameters")
display(history_df)

version,timestamp,operation,operationParameters
7,2026-04-28T12:49:47.000Z,UPDATE,"Map(predicate -> [""(ProductID#21319L = 771)""])"
6,2026-04-28T12:49:43.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)"
5,2026-04-28T12:46:13.000Z,UPDATE,"Map(predicate -> [""(ProductID#17869L = 771)""])"
4,2026-04-28T12:46:04.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)"
3,2026-04-28T12:33:36.000Z,UPDATE,"Map(predicate -> [""(ProductID#12480L = 771)""])"
2,2026-04-28T12:33:29.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)"
1,2026-04-28T10:38:29.000Z,UPDATE,"Map(predicate -> [""(ProductID#11372L = 771)""])"
0,2026-04-28T10:36:53.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)"


In [0]:
# Create schema and external table with data in one step
spark.sql("CREATE SCHEMA IF NOT EXISTS AdventureWorks")

spark.sql("""
    CREATE OR REPLACE TABLE AdventureWorks.ProductsExternal
    USING DELTA
    AS SELECT * FROM default.products_delta
""")

print("✓ Schema and external table created")
spark.sql("SELECT * FROM AdventureWorks.ProductsExternal LIMIT 5").show()

✓ Schema and external table created
+---------+--------------------+--------------+---------+
|ProductID|         ProductName|      Category|ListPrice|
+---------+--------------------+--------------+---------+
|      771|Mountain-100 Silv...|Mountain Bikes| 3059.991|
|      772|Mountain-100 Silv...|Mountain Bikes|  3399.99|
|      773|Mountain-100 Silv...|Mountain Bikes|  3399.99|
|      774|Mountain-100 Silv...|Mountain Bikes|  3399.99|
|      775|Mountain-100 Blac...|Mountain Bikes|  3374.99|
+---------+--------------------+--------------+---------+



In [0]:
# Create managed Delta table in AdventureWorks schema
df.write.format("delta").mode("overwrite").saveAsTable("AdventureWorks.ProductsManaged")

print("✓ Managed table created: AdventureWorks.ProductsManaged")

✓ Managed table created: AdventureWorks.ProductsManaged


In [0]:
# Query the managed Delta table
display(spark.sql("SELECT * FROM AdventureWorks.ProductsManaged LIMIT 10"))

ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3399.99
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99
776,"Mountain-100 Black, 42",Mountain Bikes,3374.99
777,"Mountain-100 Black, 44",Mountain Bikes,3374.99
778,"Mountain-100 Black, 48",Mountain Bikes,3374.99
779,"Mountain-200 Silver, 38",Mountain Bikes,2319.99
780,"Mountain-200 Silver, 42",Mountain Bikes,2319.99


In [0]:
# Show all tables in AdventureWorks schema
display(spark.sql("SHOW TABLES IN AdventureWorks"))

database,tableName,isTemporary
adventureworks,products,false
adventureworks,productsexternal,false
adventureworks,productsmanaged,false


In [0]:
# Compare external vs managed table metadata
print("=== External Table ===")
external_detail = spark.sql("DESCRIBE DETAIL AdventureWorks.ProductsExternal")
display(external_detail.select("format", "name", "numFiles", "sizeInBytes", "properties"))

print("\n=== Managed Table ===")
managed_detail = spark.sql("DESCRIBE DETAIL AdventureWorks.ProductsManaged")
display(managed_detail.select("format", "name", "numFiles", "sizeInBytes", "properties"))

=== External Table ===


format,name,numFiles,sizeInBytes,properties
delta,workspace.adventureworks.productsexternal,1,4367,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)"



=== Managed Table ===


format,name,numFiles,sizeInBytes,properties
delta,workspace.adventureworks.productsmanaged,1,4361,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)"


In [0]:
# Drop the demonstration tables
spark.sql("DROP TABLE IF EXISTS AdventureWorks.ProductsExternal")
spark.sql("DROP TABLE IF EXISTS AdventureWorks.ProductsManaged")

print("✓ Tables dropped")
display(spark.sql("SHOW TABLES IN AdventureWorks"))

✓ Tables dropped


database,tableName,isTemporary
adventureworks,products,false


In [0]:
# Create Products table for optimization demonstration
spark.sql("""
    CREATE OR REPLACE TABLE AdventureWorks.Products
    USING DELTA
    AS SELECT * FROM default.products_delta
""")

print("✓ Products table created")
display(spark.sql("SELECT * FROM AdventureWorks.Products LIMIT 5"))

✓ Products table created


ProductID,ProductName,Category,ListPrice
771,"Mountain-100 Silver, 38",Mountain Bikes,3059.991
772,"Mountain-100 Silver, 42",Mountain Bikes,3399.99
773,"Mountain-100 Silver, 44",Mountain Bikes,3399.99
774,"Mountain-100 Silver, 48",Mountain Bikes,3399.99
775,"Mountain-100 Black, 38",Mountain Bikes,3374.99


In [0]:
# OPTIMIZE and VACUUM demonstration
# Note: These operations are most beneficial for large tables with many files
# This small dataset (293 rows) is for demonstration purposes only

print("Running OPTIMIZE...")
spark.sql("OPTIMIZE AdventureWorks.Products")

print("Running VACUUM...")
spark.sql("VACUUM AdventureWorks.Products RETAIN 168 HOURS")

print("✓ Optimization complete")

Running OPTIMIZE...
Running VACUUM...
✓ Optimization complete
